# Nolan Gorman | Cardinals Player Valuation

A guided analysis of a 2027 tender decision. Saved outputs show the results; the source CSVs are not included in this portfolio.

## 0. Decision frame

**Player:** Nolan Gorman

**Decision:** Tender or non-tender for the 2027 season.

**Primary question:** Is the selected player's expected 2027 value greater than his projected salary and the opportunity cost of retaining his roster spot?

**Unit of analysis:** One player-season decision.

**Decision horizon:** 2027, while considering 2028 roster flexibility and internal hitting development.

## 1. Imports and notebook settings

Use the smallest practical dependency set. The initial workflow should only require standard Python plus `pandas`; add other packages only when there is a clear need.

In [7]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PLAYER_MLBAM_ID = 669357

## 2. Source files and provenance

The main public sources for this analysis are:

- `data/fangraphs_2026_stl_hitters_actuals_clean.csv` — 2026 Cardinals hitter outcomes.
- `data/fangraphs_2027_stl_hitters_projections_zips_clean.csv` — 2027 ZiPS projection.
- `data/fangraphs_2026_stl_hitters_projections_zips_clean.csv` — 2026 ZiPS projection, useful for retrospective comparison.
- `data/fangraphs_2026_stl_hitters_projections_zips_dc_clean.csv` — 2026 ZiPS DC projection, useful for retrospective comparison.
- `data/player_index.csv` — player identity and file coverage map.
- `data/contract_research.csv` — contract, arbitration, and salary research.
- `data/service_time_reconciliation_2026_with_opening_status.csv` — service-time and roster-status context.
- `data/player_evaluation.csv` — decision-workspace fields.
- `data/player_universe.csv` — broader Cardinals player universe.
- `data/projection_inputs.csv` — projection-source notes and inputs, where applicable.
- `data/raw/baseballsavant_2026_MLB_hitters_summary.csv` — optional Statcast context.

In [8]:
# Find the project directory without depending on the notebook's launch directory.
project_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_DIR = next(
    path for path in project_candidates
    if (path / "data").is_dir() and (path / "README.md").is_file()
)
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

source_files = {
    "pitcher_actuals_2026": "data/fangraphs_2026_stl_pitchers_actuals_clean.csv",
    "hitter_actuals_2026": "data/fangraphs_2026_stl_hitters_actuals_clean.csv",
    "pitcher_zips_2026": "data/fangraphs_2026_stl_pitchers_projections_zips_clean.csv",
    "hitter_zips_2026": "data/fangraphs_2026_stl_hitters_projections_zips_clean.csv",
    "pitcher_zips_dc_2026": "data/fangraphs_2026_stl_pitchers_projections_zips_dc_clean.csv",
    "hitter_zips_dc_2026": "data/fangraphs_2026_stl_hitters_projections_zips_dc_clean.csv",
    "pitcher_zips_2027": "data/fangraphs_2027_stl_pitchers_projections_zips_clean.csv",
    "hitter_zips_2027": "data/fangraphs_2027_stl_hitters_projections_zips_clean.csv",
    "player_index": "data/player_index.csv",
    "contract_research": "data/contract_research.csv",
    "player_evaluation": "data/player_evaluation.csv",
    "player_universe": "data/player_universe.csv",
    "projection_inputs": "data/projection_inputs.csv",
    "service_time": "data/service_time_reconciliation_2026_with_opening_status.csv",
    "roster_snapshot": "data/api_roster_snapshot_2026-10-12.csv",
    "opening_status": "data/opening_status_2026-03-26.csv",
    "savant_pitchers_2026": "data/raw/baseballsavant_2026_MLB_pitchers_summary.csv",
    "savant_hitters_2026": "data/raw/baseballsavant_2026_MLB_hitters_summary.csv",
}

def read_source_csv(relative_path):
    """Read a source CSV without changing the saved source file."""
    return pd.read_csv(PROJECT_DIR / relative_path, encoding="utf-8-sig")

tables = {name: read_source_csv(path) for name, path in source_files.items()}
table_shapes = pd.DataFrame(
    [
        {"table": name, "rows": frame.shape[0], "columns": frame.shape[1]}
        for name, frame in tables.items()
    ]
).sort_values("table")
table_shapes

,table,rows,columns
9,contract_research,40,14
1,hitter_actuals_2026,22,66
3,hitter_zips_2026,42,45
7,hitter_zips_2027,42,45
5,hitter_zips_dc_2026,16,45
15,opening_status,41,9
0,pitcher_actuals_2026,30,61
2,pitcher_zips_2026,36,40
6,pitcher_zips_2027,36,40
4,pitcher_zips_dc_2026,21,40


## 3. Data coverage and schema checks

Before joining or interpreting anything, verify:

- row counts for every source;
- column names and data types;
- whether `MLBAMID` is present and unique where expected;
- whether players name appears in each source;
- whether projection rows represent the same player and season;
- missing values in fields used for the decision; and
- whether any records are duplicated or unexpectedly absent.

Use `MLBAMID` as the primary join key. Use player names for display and manual review, not as the primary identity key.

In [9]:
required_columns = {
    "hitter_actuals_2026": {"Name", "Team", "Season", "PA", "WAR", "MLBAMID"},
    "hitter_zips_2027": {"Name", "Team", "PA", "WAR", "MLBAMID"},
    "hitter_zips_2026": {"Name", "Team", "PA", "WAR", "MLBAMID"},
    "hitter_zips_dc_2026": {"Name", "Team", "PA", "WAR", "MLBAMID"},
    "player_index": {"MLBAMID", "Name", "Role", "ProjectionSeason", "SourceFile"},
    "contract_research": {"mlb_id", "player_name", "arbitration_status"},
    "service_time": {"mlb_id", "player_name", "service_time_2026"},
}

id_candidates = ["MLBAMID", "mlb_id", "player_id", "PlayerId"]

def find_id_column(frame):
    return next((column for column in id_candidates if column in frame.columns), None)

qc_rows = []
for name, frame in tables.items():
    id_column = find_id_column(frame)
    duplicate_ids = None
    selected_player_rows = None
    if id_column is not None:
        ids = frame[id_column].astype("string")
        duplicate_ids = int(ids.dropna().duplicated().sum())
        selected_player_rows = int((ids == str(PLAYER_MLBAM_ID)).sum())
    missing_required = sorted(required_columns.get(name, set()) - set(frame.columns))
    qc_rows.append(
        {
            "table": name,
            "id_column": id_column,
            "duplicate_id_rows": duplicate_ids,
            "selected_player_rows": selected_player_rows,
            "missing_required_columns": ", ".join(missing_required),
        }
    )

schema_qc = pd.DataFrame(qc_rows).sort_values("table")
schema_qc

,table,id_column,duplicate_id_rows,selected_player_rows,missing_required_columns
9,contract_research,mlb_id,0.0,1.0,
1,hitter_actuals_2026,MLBAMID,0.0,1.0,
3,hitter_zips_2026,MLBAMID,0.0,1.0,
7,hitter_zips_2027,MLBAMID,0.0,1.0,
5,hitter_zips_dc_2026,MLBAMID,0.0,1.0,
15,opening_status,mlb_id,0.0,1.0,
0,pitcher_actuals_2026,MLBAMID,0.0,0.0,
2,pitcher_zips_2026,MLBAMID,0.0,0.0,
6,pitcher_zips_2027,MLBAMID,0.0,0.0,
4,pitcher_zips_dc_2026,MLBAMID,0.0,0.0,


## 4. Player identity and joins

Build the players analysis row by joining the relevant tables on `MLBAMID`. Keep the source-specific columns distinguishable when names overlap.

Expected join inputs:

1. `player_index.csv` for identity and coverage.
2. 2026 actuals for descriptive performance.
3. 2027 ZiPS for the primary public projection.
4. 2026 ZiPS and ZiPS DC for retrospective projection context.
5. Contract research for salary and arbitration context.
6. Service-time/roster data for eligibility and roster context.

Do not force a join when a source lacks player. Record the missingness and explain its impact.

In [10]:
def filter_to_player(frame, player_mlbam_id=PLAYER_MLBAM_ID):
    id_column = find_id_column(frame)
    if id_column is None:
        return frame.iloc[0:0].copy()
    ids = frame[id_column].astype("string")
    return frame.loc[ids == str(player_mlbam_id)].copy()

selected_player_source_rows = {
    name: filter_to_player(frame)
    for name, frame in tables.items()
}

selected_player_coverage = pd.DataFrame(
    [
        {
            "table": name,
            "rows_for_player": frame.shape[0],
            "columns": frame.shape[1],
        }
        for name, frame in selected_player_source_rows.items()
    ]
).sort_values("table")
selected_player_coverage

,table,rows_for_player,columns
9,contract_research,1,14
1,hitter_actuals_2026,1,66
3,hitter_zips_2026,1,45
7,hitter_zips_2027,1,45
5,hitter_zips_dc_2026,1,45
15,opening_status,1,9
0,pitcher_actuals_2026,0,61
2,pitcher_zips_2026,0,40
6,pitcher_zips_2027,0,40
4,pitcher_zips_dc_2026,0,40


## 5. 2026 actual performance

Describe what happened in 2026 before discussing what is expected in 2027.

In [ ]:
actuals = tables["hitter_actuals_2026"]

selected_player_actuals = actuals.loc[
    actuals["MLBAMID"] == PLAYER_MLBAM_ID
].copy()

actual_columns = [
    "Name",
    "Team",
    "Season",
    "Age",
    "G",
    "PA",
    "AVG",
    "OBP",
    "SLG",
    "HR",
    "ISO",
    "BB%",
    "K%",
    "BABIP",
    "wOBA",
    "wRC+",
    "WAR",
    "SB",
    "CS",
    "Fld",
    "Off",
    "Def",
    "xwOBA",
    "EV",
    "Barrel%",
    "HardHit%",
]

selected_player_actuals.loc[:, actual_columns]

,Name,Team,Season,Age,G,PA,AVG,OBP,SLG,OPS,HR,ISO,BB%,K%,BABIP,wOBA,wRC+,WAR,SB,CS,Fld,Off,Def,xwOBA,EV,Barrel%,HardHit%
5,Nolan Gorman,STL,2026,26,85,302,0.184211,0.271523,0.308271,0.579794,9,0.12406,0.109272,0.324503,0.246914,0.259737,62.355069,-0.546706,0,1,0.570938,-14.872692,-0.322409,0.27775,89.342105,0.099415,0.421053


## 6. Projection comparison

Compare ZiPS and ZiPS DC when both are available. Do not treat disagreement as an error; treat it as evidence of projection uncertainty or different playing-time assumptions.

In [ ]:
comparison_sources ={
    "2026 Actual": tables["hitter_actuals_2026"],
    "2026 ZiPS": tables["hitter_zips_2026"],
    "2026 ZiPS DC": tables["hitter_zips_dc_2026"],
    "2027 ZiPS": tables["hitter_zips_2027"],
}

comparison_columns = [
    "G",
    "PA",
    "AVG",
    "OBP",
    "SLG",
    "HR",
    "ISO",
    "BB%",
    "K%",
    "BABIP",
    "wOBA",
    "wRC+",
    "WAR",
    "SB",
    "CS",
    "Fld",
    "Off",
    "Def",
]

comparison_rows = []

for source_name, frame in comparison_sources.items():
    required_columns = ["MLBAMID"] + comparison_columns
    missing_columns = sorted(set(required_columns) - set(frame.columns))

    if missing_columns:
        raise KeyError(
            f"{source_name} is missing these columns: {missing_columns}"
        )
    player_row = frame.loc[
        frame["MLBAMID"].astype("string").eq(str(PLAYER_MLBAM_ID)),
        required_columns,
    ].copy()

    if len(player_row) != 1:
        raise ValueError(
            f"{source_name} returned {len(player_row)} rows for MLBAM ID "
            f"{PLAYER_MLBAM_ID}; expected 1."
        )

    player_row.insert(0, "Source", source_name)
    comparison_rows.append(player_row)

projection_comparison = pd.concat(
    comparison_rows,
    ignore_index=True,
).drop(columns="MLBAMID")

projection_comparison

,Source,G,PA,AVG,OBP,SLG,HR,ISO,BB%,K%,BABIP,wOBA,wRC+,WAR,SB,CS,Fld,Off,Def
0,2026 Actual,85,302,0.184211,0.271523,0.308271,9,0.124060,0.109272,0.324503,0.246914,0.259737,62.355069,-0.546706,0,1,0.570938,-14.872692,-0.322409
1,2026 ZiPS,125,465,0.224638,0.303226,0.425121,21,0.200483,0.101075,0.318280,0.290323,0.315092,103.581000,1.992910,3,1,0.800000,1.504073,2.508554
2,2026 ZiPS DC,123,532,0.224638,0.303226,0.425121,24,0.200483,0.101075,0.318280,0.290323,0.315092,103.581000,1.693580,3,1,0.632448,1.720791,-2.867552
3,2027 ZiPS,127,474,0.229858,0.308017,0.436019,22,0.206161,0.101266,0.308017,0.291829,0.318881,106.177000,2.200460,3,1,1.000000,3.000229,2.741623


## 7. Organization-wide 2027 ZiPS hitter comparison

Compare the selected players 2027 ZiPS projection with every hitter included in the 2027 ZiPS export.

The table includes roster and contract context when available. Projection-only players remain in the table but should not automatically be treated as realistic internal replacements.

Gap calculations use:

candidate projection − baseline player projection

For metrics where lower is better, a negative gap favors the candidate.

In [13]:
roster = tables["player_index"].copy()
zips_2027 = tables["hitter_zips_2027"].copy()
contracts = tables["contract_research"].copy()

# Remove hidden BOM characters and standardize ID fields.
for frame in [roster, zips_2027]:
    frame.columns = (
        frame.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )
    frame["MLBAMID"] = frame["MLBAMID"].astype("string").str.strip()

contracts = contracts.rename(columns={"mlb_id": "MLBAMID"})
contracts["MLBAMID"] = (
    contracts["MLBAMID"]
    .astype("string")
    .str.strip()
)

# These are the projection metrics used in the comparison.
projection_metric_map = {
    "G": "G",
    "PA": "PA",
    "HR": "HR",
    "SB": "SB",
    "CS": "CS",
    "AVG": "AVG",
    "OBP": "OBP",
    "SLG": "SLG",
    "ISO": "ISO",
    "BB%": "BB_pct",
    "K%": "K_pct",
    "BABIP": "BABIP",
    "wOBA": "wOBA",
    "wRC+": "wRC_plus",
    "Spd": "Spd",
    "BsR": "BsR",
    "wSB": "wSB",
    "Fld": "Fld",
    "Off": "Off",
    "Def": "Def",
    "WAR": "WAR",
}

source_projection_columns = [
    "MLBAMID",
    "Name",
    "Team",
] + list(projection_metric_map.keys())

zips_2027_selected = zips_2027[
    source_projection_columns
].rename(
    columns={
        source_name: f"ZIPS_2027_{output_name}"
        for source_name, output_name in projection_metric_map.items()
    }
)

# Keep organizational context separate from projection statistics.
roster_context = roster[
    [
        "MLBAMID",
        "Role",
        "roster_group",
        "projection_status",
        "ProjectionSeason",
        "ProjectionModel",
    ]
].copy()

contract_context = contracts[
    [
        "MLBAMID",
        "contract_bucket",
        "arbitration_status",
        "service_time_total",
        "service_time_2026",
        "MiLB_options_remaining",
        "contract_options_status",
        "confidence",
        "notes",
    ]
].rename(
    columns={
        "confidence": "contract_confidence",
        "notes": "contract_notes",
    }
)

hitter_comparison = (
    zips_2027_selected
    .merge(
        roster_context,
        on="MLBAMID",
        how="left",
        validate="many_to_one",
    )
    .merge(
        contract_context,
        on="MLBAMID",
        how="left",
        validate="many_to_one",
    )
)

numeric_projection_columns = [
    f"ZIPS_2027_{output_name}"
    for output_name in projection_metric_map.values()
]

hitter_comparison[numeric_projection_columns] = (
    hitter_comparison[numeric_projection_columns]
    .apply(pd.to_numeric, errors="coerce")
)

In [14]:
baseline_player_id = str(PLAYER_MLBAM_ID)

baseline_player_row = hitter_comparison.loc[
    hitter_comparison["MLBAMID"].eq(baseline_player_id)
]

if len(baseline_player_row) != 1:
    raise ValueError("Expected exactly one baseline player row.")

baseline_player_row = baseline_player_row.iloc[0]

for output_name in projection_metric_map.values():
    projection_column = f"ZIPS_2027_{output_name}"
    gap_column = f"Gap_{output_name}_vs_Baseline"

    hitter_comparison[gap_column] = (
        hitter_comparison[projection_column]
        - baseline_player_row[projection_column]
    )

In [15]:
gap_output_names = [
    "WAR",
    "PA",
    "G",
    "wRC_plus",
    "AVG",
    "OBP",
    "SLG",
    "HR",
    "ISO",
    "BB_pct",
    "K_pct",
    "BABIP",
    "wOBA",
    "SB",
    "CS",
    "Spd",
    "BsR",
    "wSB",
    "Fld",
    "Off",
    "Def",
]

gap_columns = [
    f"Gap_{output_name}_vs_Baseline"
    for output_name in gap_output_names
]

projection_columns = [
    f"ZIPS_2027_{output_name}"
    for output_name in gap_output_names
]

context_columns = [
    "roster_group",
    "contract_bucket",
    "arbitration_status",
    "service_time_total",
    "service_time_2026",
    "MiLB_options_remaining",
    "contract_options_status",
    "MLBAMID",
]

hitter_comparison_table = (
    hitter_comparison[
        ["Name"] + gap_columns + projection_columns + context_columns
    ]
    .sort_values(
        ["Gap_WAR_vs_Baseline", "Gap_PA_vs_Baseline"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

hitter_comparison_table

,Name,Gap_WAR_vs_Baseline,Gap_PA_vs_Baseline,Gap_G_vs_Baseline,Gap_wRC_plus_vs_Baseline,Gap_AVG_vs_Baseline,Gap_OBP_vs_Baseline,Gap_SLG_vs_Baseline,Gap_HR_vs_Baseline,Gap_ISO_vs_Baseline,Gap_BB_pct_vs_Baseline,Gap_K_pct_vs_Baseline,Gap_BABIP_vs_Baseline,Gap_wOBA_vs_Baseline,Gap_SB_vs_Baseline,Gap_CS_vs_Baseline,Gap_Spd_vs_Baseline,Gap_BsR_vs_Baseline,Gap_wSB_vs_Baseline,Gap_Fld_vs_Baseline,Gap_Off_vs_Baseline,Gap_Def_vs_Baseline,ZIPS_2027_WAR,ZIPS_2027_PA,ZIPS_2027_G,ZIPS_2027_wRC_plus,ZIPS_2027_AVG,ZIPS_2027_OBP,ZIPS_2027_SLG,ZIPS_2027_HR,ZIPS_2027_ISO,ZIPS_2027_BB_pct,ZIPS_2027_K_pct,ZIPS_2027_BABIP,ZIPS_2027_wOBA,ZIPS_2027_SB,ZIPS_2027_CS,ZIPS_2027_Spd,ZIPS_2027_BsR,ZIPS_2027_wSB,ZIPS_2027_Fld,ZIPS_2027_Off,ZIPS_2027_Def,roster_group,contract_bucket,arbitration_status,service_time_total,service_time_2026,MiLB_options_remaining,contract_options_status,MLBAMID
0,Masyn Winn,1.118290,128,15,-12.1736,0.024255,0.005316,-0.050279,-9,-0.074534,-0.028176,-0.125293,0.002564,-0.017771,8,3,1.78602,0.127186,0.127186,9.4,-7.621298,14.294179,3.318750,602,142,94.0034,0.254113,0.313333,0.385740,13,0.131627,0.073090,0.182724,0.294393,0.301110,11,4,5.26151,-0.342499,-0.342499,10.4,-4.621068,17.035802,40man_2026-10-12,ARBITRATION,2027 ARB 1; 2028 ARB 2; 2029 ARB 3; 2030 FREE ...,3.045,1.000,3.0,NOT SHOWN,691026
1,JJ Wetherholt,0.706510,68,-2,-3.4110,0.025416,0.035156,-0.062601,-11,-0.088018,0.000210,-0.149345,-0.002355,-0.004979,10,1,1.68291,1.286806,1.286806,4.8,-0.406067,5.049853,2.906970,542,125,102.7660,0.255274,0.343173,0.373418,11,0.118143,0.101476,0.158672,0.289474,0.313902,13,2,5.15840,0.817121,0.817121,5.8,2.594162,7.791476,40man_2026-10-12,GUARANTEED,NOT APPLICABLE,1.000,1.000,3.0,NOT APPLICABLE,802139
2,Nolan Gorman,0.000000,0,0,0.0000,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,2.200460,474,127,106.1770,0.229858,0.308017,0.436019,22,0.206161,0.101266,0.308017,0.291829,0.318881,3,1,3.47549,-0.469685,-0.469685,1.0,3.000229,2.741623,40man_2026-10-12,ARBITRATION,2027 ARB 2; 2028 ARB 3; 2029 FREE AGENT,4.054,0.112,1.0,NOT SHOWN,669357
3,Leo Bernal,-0.093950,6,-8,-24.9485,0.006381,-0.008854,-0.078221,-11,-0.084601,-0.024183,-0.105934,-0.013041,-0.036419,4,2,0.42714,-0.045441,-0.045441,6.0,-14.194465,13.076720,2.106510,480,119,81.2285,0.236239,0.299163,0.357798,11,0.121560,0.077083,0.202083,0.278788,0.282462,7,3,3.90263,-0.515126,-0.515126,7.0,-11.194236,15.818342,40man_2026-10-12,PRE_ARBITRATION,2027 PRE-ARB; 2028 PRE-ARB; 2029 PRE-ARB; 2030...,0.027,0.027,2.0,NOT SHOWN,699024
4,Jimmy Crooks,-0.146690,-20,-16,-18.6787,0.014103,-0.005589,-0.056792,-11,-0.070895,-0.032984,-0.041497,0.023960,-0.027267,-3,-1,-0.83986,-0.170677,-0.170677,3.0,-10.367550,9.599059,2.053770,454,111,87.4983,0.243961,0.302428,0.379227,11,0.135266,0.068282,0.266520,0.315789,0.291614,0,0,2.63563,-0.640362,-0.640362,4.0,-7.367320,12.340682,40man_2026-10-12,PRE_ARBITRATION,PRE-ARB,0.125,0.094,2.0,NOT SHOWN,699625
5,Victor Scott II,-0.254010,38,8,-23.9913,0.001084,0.000134,-0.101938,-14,-0.103022,-0.019235,-0.100986,-0.009930,-0.035022,33,5,3.76263,4.393975,4.393975,6.0,-9.886124,6.139624,1.946450,512,135,82.1857,0.230942,0.308151,0.334081,8,0.103139,0.082031,0.207031,0.281899,0.283859,36,6,7.23812,3.924290,3.924290,7.0,-6.885895,8.881246,40man_2026-10-12,PRE_ARBITRATION,2027 potential Super Two; standard projection ...,2.012,0.101,1.0,NOT SHOWN,687363
6,Iván Herrera,-0.604500,6,-7,14.2270,0.043089,0.056566,-0.001236,-7,-0.044325,0.007067,-0.112184,0.025323,0.020769,5,0,0.82116,0.841560,0.841560,-1.0,8.979466,-15.087302,1.595960,480,120,120.4040,0.272947,0.364583,0.434783,15,0.161836,0.108333,0.195833,0.317152,0.339650,8,1,4.29665,0.371875,0.371875,0.0,11.979696,-12.345679,40man_2026-10-12,ARBITRATION,2027 ARB 1; 2028 ARB 2; 2029 ARB 3; 2030 FREE ...,3.034,1.000,0.0,NOT SHOWN,671056
7,Joshua Báez,-0.641860,16,-4,-10.3244,0.008674,0.004866,-0.041524,-8,-0.050198,-0.021674,-0.024344,0.021760

### Decision-oriented comparison view

This narrower table highlights the metrics most relevant to evaluating players projected value and potential internal alternatives. The full master table remains available above for reference.

In [17]:
decision_columns = [
    "Name",
    "Role",
    "Gap_WAR_vs_Baseline",
    "Gap_PA_vs_Baseline",
    "Gap_G_vs_Baseline",
    "Gap_wRC_plus_vs_Baseline",
    "Gap_OBP_vs_Baseline",
    "Gap_ISO_vs_Baseline",
    "Gap_HR_vs_Baseline",
    "ZIPS_2027_WAR",
    "ZIPS_2027_PA",
    "ZIPS_2027_G",
    "ZIPS_2027_wRC_plus",
    "ZIPS_2027_OBP",
    "ZIPS_2027_ISO",
    "ZIPS_2027_HR",
    "ZIPS_2027_Fld",
    "ZIPS_2027_Off",
    "ZIPS_2027_Def",
    "roster_group",
    "contract_bucket",
    "arbitration_status",
    "service_time_total",
    "service_time_2026",
    "MiLB_options_remaining",
    "contract_options_status",
    "contract_confidence",
    "MLBAMID",
]

decision_view = (
    hitter_comparison[decision_columns]
    .sort_values(
        ["Gap_WAR_vs_Baseline", "Gap_PA_vs_Baseline"],
        ascending=[False, False],
        na_position="last",
    )
    .reset_index(drop=True)
)

decision_view

,Name,Role,Gap_WAR_vs_Baseline,Gap_PA_vs_Baseline,Gap_G_vs_Baseline,Gap_wRC_plus_vs_Baseline,Gap_OBP_vs_Baseline,Gap_ISO_vs_Baseline,Gap_HR_vs_Baseline,ZIPS_2027_WAR,ZIPS_2027_PA,ZIPS_2027_G,ZIPS_2027_wRC_plus,ZIPS_2027_OBP,ZIPS_2027_ISO,ZIPS_2027_HR,ZIPS_2027_Fld,ZIPS_2027_Off,ZIPS_2027_Def,roster_group,contract_bucket,arbitration_status,service_time_total,service_time_2026,MiLB_options_remaining,contract_options_status,contract_confidence,MLBAMID
0,Masyn Winn,HITTER,1.118290,128,15,-12.1736,0.005316,-0.074534,-9,3.318750,602,142,94.0034,0.313333,0.131627,13,10.4,-4.621068,17.035802,40man_2026-10-12,ARBITRATION,2027 ARB 1; 2028 ARB 2; 2029 ARB 3; 2030 FREE ...,3.045,1.000,3.0,NOT SHOWN,medium,691026
1,JJ Wetherholt,HITTER,0.706510,68,-2,-3.4110,0.035156,-0.088018,-11,2.906970,542,125,102.7660,0.343173,0.118143,11,5.8,2.594162,7.791476,40man_2026-10-12,GUARANTEED,NOT APPLICABLE,1.000,1.000,3.0,NOT APPLICABLE,medium,802139
2,Nolan Gorman,HITTER,0.000000,0,0,0.0000,0.000000,0.000000,0,2.200460,474,127,106.1770,0.308017,0.206161,22,1.0,3.000229,2.741623,40man_2026-10-12,ARBITRATION,2027 ARB 2; 2028 ARB 3; 2029 FREE AGENT,4.054,0.112,1.0,NOT SHOWN,medium,669357
3,Leo Bernal,HITTER,-0.093950,6,-8,-24.9485,-0.008854,-0.084601,-11,2.106510,480,119,81.2285,0.299163,0.121560,11,7.0,-11.194236,15.818342,40man_2026-10-12,PRE_ARBITRATION,2027 PRE-ARB; 2028 PRE-ARB; 2029 PRE-ARB; 2030...,0.027,0.027,2.0,NOT SHOWN,medium,699024
4,Jimmy Crooks,HITTER,-0.146690,-20,-16,-18.6787,-0.005589,-0.070895,-11,2.053770,454,111,87.4983,0.302428,0.135266,11,4.0,-7.367320,12.340682,40man_2026-10-12,PRE_ARBITRATION,PRE-ARB,0.125,0.094,2.0,NOT SHOWN,low,699625
5,Victor Scott II,HITTER,-0.254010,38,8,-23.9913,0.000134,-0.103022,-14,1.946450,512,135,82.1857,0.308151,0.103139,8,7.0,-6.885895,8.881246,40man_2026-10-12,PRE_ARBITRATION,2027 potential Super Two; standard projection ...,2.012,0.101,1.0,NOT SHOWN,low,687363
6,Iván Herrera,HITTER,-0.604500,6,-7,14.2270,0.056566,-0.044325,-7,1.595960,480,120,120.4040,0.364583,0.161836,15,0.0,11.979696,-12.345679,40man_2026-10-12,ARBITRATION,2027 ARB 1; 2028 ARB 2; 2029 ARB 3; 2030 FREE ...,3.034,1.000,0.0,NOT SHOWN,medium,671056
7,Joshua Báez,HITTER,-0.641860,16,-4,-10.3244,0.004866,-0.050198,-8,1.558600,490,123,95.8526,0.312883,0.155963,14,4.5,-0.162598,-0.901235,40man_2026-10-12,PRE_ARBITRATION,PRE-ARB,0.044,0.044,2.0,NOT SHOWN,low,695491
8,Thomas Saggese,HITTER,-0.877060,74,8,-19.7427,-0.002715,-0.078671,-10,1.323400,548,135,86.4343,0.305302,0.127490,12,2.3,-9.605199,4.313521,40man_2026-10-12,PRE_ARBITRATION,PRE-ARB,1.073,0.101,1.0,NOT SHOWN,low,695336
9,Alec Burleson,HITTER,-0.939740,55,13,2.3630,0.021528,-0.046078,-5,1.260720,529,140,108.5400,0.329545,0.160083,17,-0.7,5.146092,-10.418548,40man_2026-10-12,ARBITRATION,2027 ARB 2; 2028 ARB 3; 2029 FREE AGENT,4.029,1.000,3.0,NOT SHOWN,medium,676475


## 8. Salary, arbitration, and opportunity cost

Compare expected player value with expected cost.

### Salary assumptions

- 2026 Salary: $2.655 million
- 2027 status: Arbitration Year 2
- Just Baseball projects a $3.155 million and non-tender

## 9. Player interpretation

**What the evidence says:**

- Gorman’s 2026 performance was poor: .184/.272/.308, 62 wRC+, -0.55 WAR, and -14.9 offensive runs in 85 games.
- His contact metrics were mixed rather than uniformly poor, but a 32.5% strikeout rate and .124 ISO prevented that contact from becoming consistent production.
- His defense was essentially neutral, with +0.57 Fld and -0.32 Def.
- The 2027 ZiPS projection expects a major rebound: 2.20 WAR, 106 wRC+, and 22 home runs.

**What the evidence does not establish:**

- The 2027 projection does not prove that Gorman will regain his previous performance level.
- His .278 xwOBA suggests some underlying offensive ability, but it does not erase the poor 2026 results.
- Public data do not establish whether his approach, confidence, demotion, or Triple-A struggles are fixable.

**Roster and organizational context:**

- Gorman has one option remaining, providing some flexibility, but sending a player with his projected salary to Triple-A would still be inefficient.
- The Cardinals already have several left-handed or switch-hitting bats, reducing the need for another left-handed hitter.
- Thomas Saggese could receive more regular infield playing time along with Fermín
- Jesús Báez, Deniel Ortiz, and potentially Raniel Rodriguez provide additional internal alternatives.
- Non-tendering Gorman would create a 40-man roster spot for Rule 5-eligible prospects who need protection.

**Salary interpretation:**

- Gorman is entering Arbitration Year 2, with a public salary estimate of approximately $3.2 million.
- That price is significant for a player whose 2026 performance was below replacement level and whose 2027 value depends heavily on a rebound.
- The Cardinals would be paying for projected upside while accepting meaningful performance and roster risk.

**Recommendation:**

- **Non-tender Gorman.** His projection creates a legitimate rebound case, but the salary, lineup redundancy, roster opportunity cost, and available internal alternatives make the risk too high.